In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

URL = "https://www.manipalcigna.com/diet/fever-diet"
OUTPUT_FILE = "fever_diet_foods_to_avoid.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [2]:
def clean_text(value):
    """Collapse whitespace while preserving spaces between words."""
    return re.sub(r"\s+", " ", str(value or "")).strip()

In [3]:
response = requests.get(URL, headers=HEADERS, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("Page downloaded successfully.")
print("Page title:", clean_text(soup.title.get_text()) if soup.title else "Not found")

Page downloaded successfully.
Page title: Fever Diet: Indian Foods for Comfort and Recovery


In [ ]:
food_items = []

target_heading = None

for heading in soup.find_all(re.compile(r"^h[1-6]$")):
    heading_text = clean_text(heading.get_text(" ", strip=True)).lower()

    if "foods to avoid" in heading_text and "fever" in heading_text:
        target_heading = heading
        break

if target_heading is None:
    raise ValueError(
        "Could not find the 'Foods to avoid on Fever Diet' heading. "
        "The webpage structure or heading text may have changed."
    )

for element in target_heading.find_all_next():
    if element.name and re.fullmatch(r"h[1-6]", element.name):
        break

    if element.name == "li":
        item = clean_text(element.get_text(" ", strip=True))
        if item:
            food_items.append(item)

print("Avoid-list entries extracted:", len(food_items))

Avoid-list entries extracted: 5


In [5]:
seen = set()
unique_items = []

for item in food_items:
    key = item.casefold()
    if key not in seen:
        seen.add(key)
        unique_items.append(item)

df = pd.DataFrame({"food_item": unique_items})

print("Unique avoid-list entries:", len(df))
display(df)

Unique avoid-list entries: 5


,food_item
0,Fried snacks like samosas or pakoras
1,Very spicy curries or HRB masala heavy dishes
2,Sugary beverages and packaged desserts
3,Caffeinated tea or coffee in large amounts
4,Alcoholic drinks and carbonated sodas


In [6]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} entries to: {OUTPUT_FILE}")
print("CSV columns:", df.columns.tolist())

Saved 5 entries to: fever_diet_foods_to_avoid.csv
CSV columns: ['food_item']
